### **Caesar Cipher and Automatic Brute Force**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Caesar_Cipher.ipynb)

> The **Caesar cipher** shifts every letter by a fixed number of positions. With only 26 possible keys, it falls to **brute force** in microseconds. This notebook implements the cipher, then breaks it **automatically** by scoring each candidate with English letter frequencies (chi-squared), instead of reading 26 outputs by eye.

> [!WARNING]
> Educational code. Never use these implementations to protect real data: use a vetted library (for example, `cryptography`) with modern, authenticated algorithms.

In [1]:
# Encryption and decryption: letters are shifted, everything else (spaces, digits, punctuation) is kept
import string

ALPHABET = string.ascii_uppercase   # 26 letters

def shift_text(text, key):
    result = []
    for char in text.upper():
        if char in ALPHABET:
            result.append(ALPHABET[(ALPHABET.index(char) + key) % 26])
        else:
            result.append(char)            # keep non-letters unchanged
    return "".join(result)

def caesar_encrypt(text, key):
    return shift_text(text, key)

def caesar_decrypt(text, key):
    return shift_text(text, -key)

message = "Python for Cryptography, 2026!"
encrypted = caesar_encrypt(message, 6)
print("Plaintext :", message)
print("Encrypted :", encrypted)
print("Decrypted :", caesar_decrypt(encrypted, 6))
assert caesar_decrypt(encrypted, 6) == message.upper()

Plaintext : Python for Cryptography, 2026!
Encrypted : VEZNUT LUX IXEVZUMXGVNE, 2026!
Decrypted : PYTHON FOR CRYPTOGRAPHY, 2026!


#### **Brute Force with Automatic Scoring**

Each of the 26 keys is tried. The **chi-squared** statistic compares the letter counts of each candidate with the expected English frequencies: the lower the score, the more English-like the text. The best score reveals the key with no human reading.

In [2]:
# Try every key and rank the candidates by how English-like they are
ENGLISH_FREQ = {  # relative frequency of letters in English text (%)
    "E": 12.7, "T": 9.1, "A": 8.2, "O": 7.5, "I": 7.0, "N": 6.7, "S": 6.3, "H": 6.1, "R": 6.0,
    "D": 4.3, "L": 4.0, "C": 2.8, "U": 2.8, "M": 2.4, "W": 2.4, "F": 2.2, "G": 2.0, "Y": 2.0,
    "P": 1.9, "B": 1.5, "V": 1.0, "K": 0.8, "J": 0.15, "X": 0.15, "Q": 0.1, "Z": 0.07,
}

def chi_squared(text):
    letters = [c for c in text if c in ALPHABET]
    total = len(letters) or 1
    return sum((letters.count(l) - total * f / 100) ** 2 / (total * f / 100) for l, f in ENGLISH_FREQ.items())

def crack_caesar(ciphertext):
    candidates = [(chi_squared(caesar_decrypt(ciphertext, k)), k, caesar_decrypt(ciphertext, k)) for k in range(26)]
    return sorted(candidates)

secret = caesar_encrypt("Attack at dawn, the enemy moves through the northern pass", 17)
ranking = crack_caesar(secret)
print("Ciphertext:", secret, "\n")
for score, key, text in ranking[:3]:
    print(f"key={key:2}  score={score:7.1f}  {text}")
print(f"\nRecovered key: {ranking[0][1]}")
assert ranking[0][1] == 17

Ciphertext: RKKRTB RK URNE, KYV VEVDP DFMVJ KYIFLXY KYV EFIKYVIE GRJJ 

key=17  score=   14.1  ATTACK AT DAWN, THE ENEMY MOVES THROUGH THE NORTHERN PASS
key= 2  score=  112.6  PIIPRZ PI SPLC, IWT TCTBN BDKTH IWGDJVW IWT CDGIWTGC EPHH
key= 3  score=  133.4  OHHOQY OH ROKB, HVS SBSAM ACJSG HVFCIUV HVS BCFHVSFB DOGG

Recovered key: 17


#### **Key Takeaways**

* A key space of 26 is no protection: brute force is instant.
* Letter frequencies survive substitution, so scoring candidates automates the attack (frequency analysis).
* Modern ciphers need key spaces of at least 2^128 and must hide statistical patterns of the plaintext.